In [2]:
import pandas as pd

df = pd.read_csv("data/final_retail_demand_results.csv")

print(df.shape)
print(df.head())

(972, 7)
   Actual_Demand  Predicted_Demand      Error  Demand_Status Stock_Risk  \
0           29.0         45.633333 -16.633333  Normal Demand       Safe   
1           65.0         45.233333  19.766667  Normal Demand       Safe   
2           52.0         50.206667   1.793333  Normal Demand       Safe   
3            7.0         52.780000 -45.780000  Normal Demand       Safe   
4           36.0         41.396667  -5.396667  Normal Demand       Safe   

   Inventory_Level Recommended_Action  
0             83.0         No Reorder  
1            283.0         No Reorder  
2             89.0         No Reorder  
3            122.0         No Reorder  
4            292.0         No Reorder  


In [3]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Missing values:
Actual_Demand         0
Predicted_Demand      0
Error                 0
Demand_Status         0
Stock_Risk            0
Inventory_Level       0
Recommended_Action    0
dtype: int64

Duplicate rows:
0


In [4]:
df["Calculated_Error"] = df["Actual_Demand"] - df["Predicted_Demand"]

print(df[["Actual_Demand", "Predicted_Demand", "Error", "Calculated_Error"]].head(10))

   Actual_Demand  Predicted_Demand      Error  Calculated_Error
0           29.0         45.633333 -16.633333        -16.633333
1           65.0         45.233333  19.766667         19.766667
2           52.0         50.206667   1.793333          1.793333
3            7.0         52.780000 -45.780000        -45.780000
4           36.0         41.396667  -5.396667         -5.396667
5            7.0         43.640000 -36.640000        -36.640000
6           77.0         55.056667  21.943333         21.943333
7           50.0         44.703333   5.296667          5.296667
8           72.0         46.133333  25.866667         25.866667
9           71.0         62.480000   8.520000          8.520000


In [5]:
difference = (df["Error"] - df["Calculated_Error"]).abs()

print("Maximum difference:", difference.max())

Maximum difference: 1.1368683772161603e-13


In [6]:
print(df["Demand_Status"].value_counts())
print()
print(df["Stock_Risk"].value_counts())
print()
print(df["Recommended_Action"].value_counts())

Demand_Status
Normal Demand    952
High Demand       20
Name: count, dtype: int64

Stock_Risk
Safe    872
Risk    100
Name: count, dtype: int64

Recommended_Action
No Reorder    872
Reorder       100
Name: count, dtype: int64


In [7]:
mismatch = df[
    ((df["Stock_Risk"] == "Risk") & (df["Recommended_Action"] != "Reorder")) |
    ((df["Stock_Risk"] == "Safe") & (df["Recommended_Action"] != "No Reorder"))
]

print("Logic mismatches:", len(mismatch))

Logic mismatches: 0


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(df["Actual_Demand"], df["Predicted_Demand"])
rmse = np.sqrt(mean_squared_error(df["Actual_Demand"], df["Predicted_Demand"]))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 29.146008230452676
RMSE: 129.468885229766


In [9]:
print("Actual Demand Range:")
print("Minimum:", df["Actual_Demand"].min())
print("Maximum:", df["Actual_Demand"].max())

print("\nPredicted Demand Range:")
print("Minimum:", df["Predicted_Demand"].min())
print("Maximum:", df["Predicted_Demand"].max())

Actual Demand Range:
Minimum: 6.0
Maximum: 3750.0

Predicted Demand Range:
Minimum: 26.35
Maximum: 531.25


In [10]:
df["Absolute_Error"] = abs(df["Actual_Demand"] - df["Predicted_Demand"])

high_errors = df.sort_values("Absolute_Error", ascending=False).head(10)

print(high_errors[
    ["Actual_Demand", "Predicted_Demand", "Error", "Absolute_Error", "Demand_Status"]
])

     Actual_Demand  Predicted_Demand        Error  Absolute_Error  \
271         3750.0         47.790000  3702.210000     3702.210000   
34          1000.0         58.813333   941.186667      941.186667   
347          800.0         71.763333   728.236667      728.236667   
611           15.0        531.250000  -516.250000      516.250000   
944           58.0        301.803333  -243.803333      243.803333   
210           70.0        258.613333  -188.613333      188.613333   
945           26.0        213.716667  -187.716667      187.716667   
934           85.0        248.936667  -163.936667      163.936667   
488           22.0        172.370000  -150.370000      150.370000   
615           52.0        198.453333  -146.453333      146.453333   

     Demand_Status  
271  Normal Demand  
34   Normal Demand  
347    High Demand  
611    High Demand  
944    High Demand  
210    High Demand  
945    High Demand  
934    High Demand  
488    High Demand  
615    High Demand  


In [11]:
pd.set_option("display.max_columns", None)

print(df.loc[[271, 34, 347, 611]])

     Actual_Demand  Predicted_Demand        Error  Demand_Status Stock_Risk  \
271         3750.0         47.790000  3702.210000  Normal Demand       Safe   
34          1000.0         58.813333   941.186667  Normal Demand       Safe   
347          800.0         71.763333   728.236667    High Demand       Risk   
611           15.0        531.250000  -516.250000    High Demand       Risk   

     Inventory_Level Recommended_Action  Calculated_Error  Absolute_Error  
271            263.0         No Reorder       3702.210000     3702.210000  
34             217.0         No Reorder        941.186667      941.186667  
347             26.0            Reorder        728.236667      728.236667  
611             67.0            Reorder       -516.250000      516.250000  


In [12]:
print(df.groupby("Demand_Status")[["Actual_Demand", "Predicted_Demand"]].agg(["min", "max", "mean"]))

              Actual_Demand                    Predicted_Demand              \
                        min     max       mean              min         max   
Demand_Status                                                                 
High Demand             9.0   800.0  89.050000        71.763333  531.250000   
Normal Demand           6.0  3750.0  55.385504        26.350000   68.993333   

                           
                     mean  
Demand_Status              
High Demand    181.558333  
Normal Demand   46.576078  


In [13]:
print(df[df["Demand_Status"] == "Normal Demand"]["Predicted_Demand"].max())
print(df[df["Demand_Status"] == "High Demand"]["Predicted_Demand"].min())

68.99333333333334
71.76333333333334


In [14]:
print(df[df["Actual_Demand"] > 500][
    ["Actual_Demand", "Predicted_Demand", "Demand_Status"]
])

     Actual_Demand  Predicted_Demand  Demand_Status
34          1000.0         58.813333  Normal Demand
271         3750.0         47.790000  Normal Demand
347          800.0         71.763333    High Demand


In [15]:
normal_df = df[df["Actual_Demand"] <= 500]

mae_without_extreme = mean_absolute_error(
    normal_df["Actual_Demand"],
    normal_df["Predicted_Demand"]
)

rmse_without_extreme = np.sqrt(mean_squared_error(
    normal_df["Actual_Demand"],
    normal_df["Predicted_Demand"]
))

print("MAE without extreme rows:", mae_without_extreme)
print("RMSE without extreme rows:", rmse_without_extreme)

MAE without extreme rows: 23.692762297901616
RMSE without extreme rows: 34.753013122749


In [16]:
print(df["Absolute_Error"].describe())


count     972.000000
mean       29.146008
std       126.210500
min         0.003333
25%        10.487500
50%        21.213333
75%        32.384167
max      3702.210000
Name: Absolute_Error, dtype: float64
